In [10]:
import numpy as np
import pandas as pd

from data.data import CollectionAccessor, ImageHandler, EmbeddingSpaceAccessor

In [11]:
def init_DMG():
    DMG_DIR = "./data/DMG"
    image_folder = DMG_DIR+"/images"
    image_handler = ImageHandler("DMG", image_folder=image_folder, keep_prefix=False)

    time_stamp, pub_file, priv_file = CollectionAccessor.get_latest_dump(DMG_DIR+"/dumps")


    dmg_meta = dict(name="Design Museum Gent (public & private)", id_="DMG_"+time_stamp,
                creation_timestamp=time_stamp, language="nl")
    df = CollectionAccessor.get_DMG(pub_path=pub_file, #get_latest("./data/dumps", contains="public"),
                                     priv_path=priv_file, #get_latest("./data/dumps", contains="private"),
                                     rights_path=DMG_DIR+"/rights.csv",
                                     image_handler=image_handler,
                                     **dmg_meta)
    return df


dmg = init_DMG()

100%|████████████████████████████████████████████████████████████████| 24781/24781 [00:13<00:00, 1804.99it/s]


In [13]:
class Test:
    @staticmethod
    def parse_filepath(s):#     folder, file = s.rsplit("/", maxsplit=2)#[1:]

        *_, folder, file = s.rsplit("/", maxsplit=2)#[1:]
        try:
            obj_rendition, extension = file.rsplit(".", maxsplit=1)
        except ValueError:
            obj_rendition = file
            extension = None
            
        try:
            obj_num, rendition_ind =  obj_rendition.rsplit("$", maxsplit=1)
        except ValueError:
            obj_num = obj_rendition
            rendition_ind = None
        # rendition_id = ((rendition_ind[0] if rendition_ind[0] else None) if rendition_ind else None)
        return dict(path=folder, object_number=obj_num, 
                    rendition_index=rendition_ind, file_extension=extension)

    # applies directly to Series objects
    @classmethod
    def object_number_from_path(cls, path_series):
        get_obj_num = lambda s: cls.parse_filepath(s)["object_number"]
        obj_nums = path_series.apply(get_obj_num)
        obj_nums.name = "object_number"
        return obj_nums

---
## file path parsing

In [39]:
filenames = pd.read_csv("./data/DMG/images/filenames.csv").raw
path_obj_nums = Test.object_number_from_path("./" + filenames)

filenames.index = path_obj_nums
filenames = filenames.groupby(filenames.index)

In [46]:
filenames.apply(len).value_counts()

folders = filenames.apply(lambda ls: set(s.split("/")[0] for s in ls))

x = folders[folders.apply(len) > 1].apply(list).apply(sorted)



# x.name = "folder_names"
# x = x.to_frame()

# import csv

# x["folder1"] = x.folder_names.apply(lambda ls: ls[0])
# x["folder2"] = x.folder_names.apply(lambda ls: ls[1])

# x[["folder1", "folder2"]].to_csv("object_numbers_with_multiple_folders.csv", 
#          index=True, quoting=csv.QUOTE_ALL, quotechar='"')


---
## primary image tables

In [19]:
tbl = pd.read_csv("./data/DMG/Statusdocument per objectnummer en vervaardiger - inhoudelijk.csv")
map1 = tbl[["objectnummer", "naam beeld(en)"]].dropna()
map1.columns = ["object_number", "filename"]
map1 = map1.drop_duplicates(subset="object_number")
map1 = map1.set_index("object_number")#.drop_duplicates()

tbl2 = pd.read_excel("./data/DMG/StaDoc20251029_beelden.xlsx").dropna()#.reset_index()
map2 = tbl2
map2.columns = ["object_number", "filename"]
# map2 = map2.drop_duplicates(subset="object_number")
map2 = map2.set_index("object_number")

In [ ]:
shared = sorted(set(map2.index) & set(dmg.index))

map2.loc[shared]


for o in dmg.index:
    try:
        map2.loc[o]
    except KeyError:
        

In [ ]:
map2.filename

In [ ]:
filenames[filenames.object_number == "1987-0011"]

In [ ]:
map2

---

In [ ]:
cs = map2.index.value_counts()
cs = cs[cs > 1]
for o in cs.index:
    if len(map2.loc[o].filename.unique()) > 1:
        print(o, map2.loc[o].filename.unique())

In [ ]:
# map2.to_csv("map2.csv")
# map2 = pd.read_csv("map2.csv")
# map2 = map2.drop_duplicates(subset="object_number")
# map2 = map2.set_index("object_number")#.drop_duplicates()

In [ ]:
shared_keys = sorted(set(map1.index) & set(map2.index))
map1.loc[shared_keys][(map1.loc[shared_keys] == map2.loc[shared_keys])]


In [ ]:
m = map1.join(map2, how="inner", lsuffix="1", rsuffix="2")

m[m.filename1 != m.filename2]


# m.loc["2017-0308"].filename1.replace("-", "").replace("$", "").split(";")[1]

In [ ]:
m.index.str.slice(0, 4).value_counts()

---
## ------------------------------------------------------------------------------------------

In [1]:
s = """0364                            {2023-0048_0-4, 0364}
0395                            {2023-0048_0-4, 0395}
0405                            {2023-0048_0-4, 0405}
0416                            {2023-0048_0-4, 0416}
1451_1-2                         {1452_0-2, 1451_0-2}
1876                                     {1876, 1866}
1978-0037_0-4              {1978-0031, 1978-0037_0-4}
1978-0046                {1978-0106_00-22, 1978-0046}
1978-0098                      {1978-0098, 1987-0098}
1987-0189                      {1987-0189, 1987-0188}
1987-0413                      {1987-0412, 1987-0413}
1987-0422                      {1987-0422, 1987-0421}
1987-0582                  {1987-0582, 1987-0583_0-3}
1987-0656_3-3          {1987-0655_0-3, 1987-0656_0-3}
1987-0989_0-4          {1987-0988_0-3, 1987-0989_0-4}
1987-0989_1-4          {1987-0988_0-3, 1987-0989_0-4}
1987-0989_2-4          {1987-0988_0-3, 1987-0989_0-4}
1987-0989_3-4          {1987-0988_0-3, 1987-0989_0-4}
1987-0989_4-4          {1987-0988_0-3, 1987-0989_0-4}
1987-1357_9-9              {1987-1357_0-9, 1987-1360}
1989-0038_16-17    {1989-0038_16-17, 1989-0038_00-17}
1990-0040_0-9          {1990-0040_0-9, 1990-0041_0-9}
1990-0040_1-9          {1990-0040_0-9, 1990-0041_0-9}
1990-0040_2-9          {1990-0040_0-9, 1990-0041_0-9}
1990-0040_3-9          {1990-0040_0-9, 1990-0041_0-9}
1990-0040_4-9          {1990-0040_0-9, 1990-0041_0-9}
1990-0040_5-9          {1990-0040_0-9, 1990-0041_0-9}
1990-0040_6-9          {1990-0040_0-9, 1990-0041_0-9}
1990-0040_7-9          {1990-0040_0-9, 1990-0041_0-9}
1990-0040_8-9          {1990-0040_0-9, 1990-0041_0-9}
1990-0040_9-9          {1990-0040_0-9, 1990-0041_0-9}
1995-0043_01-15          {1995-0043_00-15, 1995-0047}
1995-0043_02-15          {1995-0043_00-15, 1995-0047}
1995-0043_03-15          {1995-0043_00-15, 1995-0047}
1995-0043_04-15          {1995-0043_00-15, 1995-0047}
1995-0043_05-15          {1995-0043_00-15, 1995-0047}
1995-0043_06-15          {1995-0043_00-15, 1995-0047}
1995-0043_07-15          {1995-0043_00-15, 1995-0047}
1995-0043_08-15          {1995-0043_00-15, 1995-0047}
1995-0043_09-15          {1995-0043_00-15, 1995-0047}
1995-0043_10-15          {1995-0043_00-15, 1995-0047}
1995-0043_11-15          {1995-0043_00-15, 1995-0047}
1995-0043_12-15          {1995-0043_00-15, 1995-0047}
1995-0043_13-15          {1995-0043_00-15, 1995-0047}
1995-0043_14-15          {1995-0043_00-15, 1995-0047}
1995-0043_15-15          {1995-0043_00-15, 1995-0047}
2000-0005_0-4          {2000-0004_0-4, 2000-0005_0-4}
2001-0135                  {2001-0134_0-2, 2001-0135}
2012-0029                  {2012-0030_0-2, 2012-0029}
2017-0252            {2017-0252_ROOD, 2017-0251_ROOD}
BK9090                      {BK9090, 2018-0352_00-14}
FH_0101                      {FH-0101, FH-0010_00-36}
"""

In [9]:
import pandas as pd
data = [(l.split("  ")[0], (l.split("  ")[-1])) for l in s.split("\n")]

# pd.DataFrame(data)

data

[('0364', '{2023-0048_0-4, 0364}'),
 ('0395', '{2023-0048_0-4, 0395}'),
 ('0405', '{2023-0048_0-4, 0405}'),
 ('0416', '{2023-0048_0-4, 0416}'),
 ('1451_1-2', ' {1452_0-2, 1451_0-2}'),
 ('1876', ' {1876, 1866}'),
 ('1978-0037_0-4', '{1978-0031, 1978-0037_0-4}'),
 ('1978-0046', '{1978-0106_00-22, 1978-0046}'),
 ('1978-0098', '{1978-0098, 1987-0098}'),
 ('1987-0189', '{1987-0189, 1987-0188}'),
 ('1987-0413', '{1987-0412, 1987-0413}'),
 ('1987-0422', '{1987-0422, 1987-0421}'),
 ('1987-0582', '{1987-0582, 1987-0583_0-3}'),
 ('1987-0656_3-3', '{1987-0655_0-3, 1987-0656_0-3}'),
 ('1987-0989_0-4', '{1987-0988_0-3, 1987-0989_0-4}'),
 ('1987-0989_1-4', '{1987-0988_0-3, 1987-0989_0-4}'),
 ('1987-0989_2-4', '{1987-0988_0-3, 1987-0989_0-4}'),
 ('1987-0989_3-4', '{1987-0988_0-3, 1987-0989_0-4}'),
 ('1987-0989_4-4', '{1987-0988_0-3, 1987-0989_0-4}'),
 ('1987-1357_9-9', '{1987-1357_0-9, 1987-1360}'),
 ('1989-0038_16-17', '{1989-0038_16-17, 1989-0038_00-17}'),
 ('1990-0040_0-9', '{1990-0040_0-9, 1990-0

In [ ]:
eval